### 0. Clean repo

In [1]:
import os

DIRECTORIES = [
    "../models", 
    # "../data/raw/files",
	"../plots",
    "../tmp",
    "../results",
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


Deleted files and directories:
 - ../models/model_subject_15_hands_vs_feet__imagery.joblib
 - ../models/model_subject_42_hands_vs_feet__imagery.joblib
 - ../models/model_subject_77_hands_vs_feet__imagery.joblib
 - ../models/model_subject_82_hands_vs_feet__imagery.joblib
 - ../models/model_subject_96_hands_vs_feet__imagery.joblib
 - ../models/model_subject_97_hands_vs_feet__imagery.joblib
 - ../results/results-hands_vs_feet__imagery-20250401-201326.json


### 1. Import & load libraries

In [2]:
%pip cache purge
%pip install -r ../requirements.txt


Files removed: 0 (0 bytes)
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.python.org/simple
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import concurrent.futures

import json
import os
import time
import logging

import mne
from mne.io import concatenate_raws
from mne.io.edf import read_raw_edf
from mne.datasets import eegbci
from mne import events_from_annotations, pick_types
from mne.channels import make_standard_montage
from mne.preprocessing import ICA
from mne.decoding import SPoC

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score, cross_validate
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.base import BaseEstimator, TransformerMixin

from scipy import linalg

from joblib import dump, load

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import ShuffleSplit, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


### 2. Configuration

| Run       | Task                                |
|-----------|-------------------------------------|
| 1         | Baseline, eyes open                 |
| 2         | Baseline, eyes closed               |
| 3, 7, 11  | Motor execution: left vs right hand |
| 4, 8, 12  | Motor imagery: left vs right hand   |
| 5, 9, 13  | Motor execution: hands vs feet      |
| 6, 10, 14 | Motor imagery: hands vs feet        |

In [4]:
DATA_SAMPLE_PATH = "../data/raw/files"

EXPERIMENTS = {
    "hands_vs_feet__action": ['do/hands', 'do/feet'],
    "hands_vs_feet__imagery": ['imagine/hands', 'imagine/feet'],
    "imagery_vs_action__hands": ['do/hands', 'imagine/hands'],
    "imagery_vs_action__feets": ['do/feet', 'imagine/feet'],
}

EXPERIMENTS_IDS = {
    'action': [5, 9, 13],
    'imagery': [6, 10, 14]
}


### 3. CSP Implementation

In [5]:
class CSP(BaseEstimator, TransformerMixin):
    """
    CSP implementation based on MNE implementation
    """
    def __init__(self, n_components=6):
        self.n_components = n_components
        self.filters = None
        self.n_classes = None
        self.mean = None
        self.std = None

    def calculate_cov_(self, X, y):
        """Calculate the covariance matrices for each class."""
        _, n_channels, _ = X.shape
        covs = []

        for l in self.n_classes:
            lX = X[np.where(y == l)]
            lX = lX.transpose([1, 0, 2])
            lX = lX.reshape(n_channels, -1)
            covs.append(np.cov(lX))

        return np.asarray(covs)

    def calculate_eig_(self, covs):
        """Calculate eigenvalues and eigenvectors for pairwise combinations of covariance matrices."""
        eigenvalues, eigenvectors = [], []

        for idx, cov in enumerate(covs):
            for iidx, compCov in enumerate(covs):
                if idx < iidx:
                    eigVals, eigVects = linalg.eig(cov, cov + compCov)
                    sorted_indices = np.argsort(np.abs(eigVals - 0.5))[::-1]
                    eigenvalues.append(eigVals[sorted_indices])
                    eigenvectors.append(eigVects[:, sorted_indices])

        return eigenvalues, eigenvectors

    def pick_filters(self, eigenvectors):
        """Select CSP filters based on the sorted eigenvectors."""
        filters = []

        for EigVects in eigenvectors:
            if filters == []:
                filters = EigVects[:, :self.n_components]
            else:
                filters = np.concatenate([filters, EigVects[:, :self.n_components]], axis=1)

        self.filters = filters.T

    def fit(self, X, y):
        self.n_classes = np.unique(y)

        if len(self.n_classes) < 2:
            raise ValueError("n_classes must be >= 2")

        covs = self.calculate_cov_(X, y)
        eigenvalues, eigenvectors = self.calculate_eig_(covs)
        self.pick_filters(eigenvectors)

        X = np.asarray([np.dot(self.filters, epoch) for epoch in X])
        X = (X ** 2).mean(axis=2)

        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0)

    def transform(self, X):
        X = np.asarray([np.dot(self.filters, epoch) for epoch in X])
        X = (X ** 2).mean(axis=2)
        X -= self.mean
        X /= self.std
        return X

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)


### 4. Load data

In [6]:
def fetch_data(subjNumber):
    run_execution = [5, 9, 13]  # (open and close both fists or both feet)
    run_imagery =  [6, 10, 14]  # (imagine opening and closing both fists or both feet)

    raw_files = []

    for i, j in zip(run_execution, run_imagery):
        try:
            raw_files_execution = [read_raw_edf(f, preload=True, stim_channel='auto') for f in
                                eegbci.load_data(subjNumber, i, path=DATA_SAMPLE_PATH)]
            raw_execution = concatenate_raws(raw_files_execution)

            raw_files_imagery = [read_raw_edf(f, preload=True, stim_channel='auto') for f in
                                eegbci.load_data(subjNumber, j, path=DATA_SAMPLE_PATH)]
            raw_imagery = concatenate_raws(raw_files_imagery)

            events, _ = mne.events_from_annotations(raw_execution, event_id=dict(T0=1, T1=2, T2=3))
            mapping = {1: 'rest', 2: 'do/feet', 3: 'do/hands'}
            annot_from_events = mne.annotations_from_events(
                events=events, event_desc=mapping, sfreq=raw_execution.info['sfreq'],
                orig_time=raw_execution.info['meas_date'])
            raw_execution.set_annotations(annot_from_events)

            # Annotations for imagery
            events, _ = mne.events_from_annotations(raw_imagery, event_id=dict(T0=1, T1=2, T2=3))
            mapping = {1: 'rest', 2: 'imagine/feet', 3: 'imagine/hands'}
            annot_from_events = mne.annotations_from_events(
                events=events, event_desc=mapping, sfreq=raw_imagery.info['sfreq'],
                orig_time=raw_imagery.info['meas_date'])
            raw_imagery.set_annotations(annot_from_events)

            raw_files.append(raw_execution)
            raw_files.append(raw_imagery)
        except Exception as e:
            print(f"❌ Error processing subject {subjNumber}, run {i}/{j}: {str(e)}")
            try:
                print(f"Trying to download data for subject {subjNumber}...")
                eegbci.load_data(subjNumber, i, path=DATA_SAMPLE_PATH, force_update=True)
                eegbci.load_data(subjNumber, j, path=DATA_SAMPLE_PATH, force_update=True)
            except Exception as e2:
                print(f"❌ Error downloading data: {str(e2)}")
            
    if not raw_files:
        raise ValueError(f"Could not obtain data for subject {subjNumber}")
            
    raw = concatenate_raws(raw_files)

    event, event_dict = events_from_annotations(raw)
    picks = pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')

    return [raw, event, event_dict, picks]


def prepare_data(raw, plotIt=False):
    eegbci.standardize(raw)
    montage = make_standard_montage("biosemi64")
    raw.set_montage(montage, on_missing='ignore')

    return raw


### 5. Signal filtering

In [7]:
def filter_data(raw, plotIt=False):
    raw.filter(8, 40, fir_design='firwin', skip_by_annotation='edge')
    return raw


def filter_eye_artifacts(raw, picks, method, plotIt=False):
    raw_corrected = raw.copy()
    n_components = 20

    ica = ICA(n_components=n_components, method=method, fit_params=None)
    ica.fit(raw_corrected, picks=picks)

    [eog_indicies, scores] = ica.find_bads_eog(raw, ch_name='Fpz', threshold=1.5)
    ica.exclude.extend(eog_indicies)
    ica.apply(raw_corrected, n_pca_components=n_components, exclude=ica.exclude)

    return raw_corrected


### 6. Events extraction

In [8]:
def fetch_events(data_filtered, tmin=-1, tmax=4):
    events, event_ids = events_from_annotations(data_filtered)
    picks = mne.pick_types(data_filtered.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')
    epochs = mne.Epochs(data_filtered, events, event_ids, tmin, tmax, proj=True,
                        picks=picks, baseline=None, preload=True)
    labels = epochs.events[:, -1]
    return labels, epochs, picks


### 7. Epochs

In [9]:
def pre_process_data(subjectID, experiments):
    [raw, event, event_dict, picks] = fetch_data(subjectID)
    raw_prepared = prepare_data(raw)
    raw_filtered = filter_data(raw_prepared)
    labels, epochs, picks = fetch_events(raw_filtered)

    # Extract only the epochs corresponding to the selected labels
    selected_epochs = epochs[experiments]
    X = selected_epochs.get_data()
    y = selected_epochs.events[:, -1] - 1
    
    # Check if there's enough data
    classes, counts = np.unique(y, return_counts=True)
    print(f"Class distribution: {dict(zip(classes, counts))}")
    
    min_samples = min(counts)
    if min_samples < 10:
        print(f"❗ WARNING: Very few samples for some class (minimum {min_samples}).")
        print("   This may lead to overfitting. Consider obtaining more data.")
    
    # Check if the data is balanced
    if max(counts) > min_samples * 1.5:  # If the majority class has 50% more samples
        print("❗ WARNING: Unbalanced data. Consider balancing techniques.")

    return [X, y, epochs]



### 8. Split dataset

In [10]:
def split_data(X, y, test_size=0.2):
    """
    Split data into training and test sets with option to balance data.
    """
    print(f"Original dataset shape: {X.shape}")
    
    # Check class distribution before balancing
    classes, counts = np.unique(y, return_counts=True)
    print(f"Original class distribution: {dict(zip(classes, counts))}")
    
    # Balance the dataset
    X, y = balance_dataset(X, y)
    
    # Split: separate test set
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y
    )
    print(f"Dataset shape after balancing: {X.shape}")
    print(f"Train set shape:  {X_train.shape}")
    print(f"Test set shape:   {X_test.shape}")
    
    # Check class distribution in each split
    for name, y_set in [("Train", y_train), ("Test", y_test)]:
        classes, counts = np.unique(y_set, return_counts=True)
        print(f"{name} set class distribution: \t{dict(zip(classes, counts))}")
    
    return X_train, X_test, y_train, y_test


### 9. Balance dataset

In [11]:
def balance_dataset(X, y):
    """Balance dataset using undersampling of majority class"""
    # Find class distribution
    classes, counts = np.unique(y, return_counts=True)
    min_samples = min(counts)
    
    # If imbalanced
    if max(counts) > min_samples * 1.2:  # 20% threshold
        print(f"❗ Imbalanced dataset detected ({dict(zip(classes, counts))}). Balancing...")
        # Simple undersampling of majority class
        X_balanced, y_balanced = [], []
        for i, cls in enumerate(classes):
            idx = np.where(y == cls)[0]
            if len(idx) > min_samples:
                # Undersample the majority class
                idx = np.random.choice(idx, min_samples, replace=False)
            X_balanced.append(X[idx])
            y_balanced.append(y[idx])
        
        X_balanced = np.concatenate(X_balanced)
        y_balanced = np.concatenate(y_balanced)
        
        # Shuffle the balanced dataset
        shuffle_idx = np.random.permutation(len(y_balanced))
        X_balanced = X_balanced[shuffle_idx]
        y_balanced = y_balanced[shuffle_idx]
        
        print(f"✅ Dataset balanced: {dict(zip(np.unique(y_balanced), np.bincount(y_balanced)))}")
        return X_balanced, y_balanced
    
    print("Dataset is already balanced.")
    return X, y  # Return original if balanced


### 10. Training Pipeline with Cross Validation

#### **Classifiers with Regularization**

These models aim to improve generalization and reduce the risk of overfitting through regularization techniques.

1. ***Linear Discriminant Analysis (LDA) with Regularization***
 
``` python
        lda = LDA(solver='lsqr', shrinkage=0.7)
```

- Method: Uses lsqr as the solver with shrinkage set to 0.7.

- Regularization: Shrinkage enhances stability and accuracy, especially in high-dimensional datasets.

- Use Cases: Suitable for classification problems with normality assumptions in the data.

2. ***Logistic Regression with L1 Penalty***

``` python
        log_reg = LogisticRegression(
            penalty='l1',
            solver='saga',
            C=0.2,
            max_iter=300,
            tol=1e-4,
            multi_class='auto',
            class_weight='balanced'
        )
```

- Regularization: L1 penalty (Lasso), which induces sparsity in the coefficients.

- Solver: saga, efficient for large datasets.

- Penalty Parameter C: 0.2, meaning stronger regularization.

- Balanced Classes: class_weight='balanced' adjusts weights automatically.

3. ***Random Forest Classifier with Hyperparameter Tuning***

``` python
        rfc = RandomForestClassifier(
            n_estimators=150,
            max_depth=4,
            min_samples_split=8,
            min_samples_leaf=4,
            class_weight='balanced_subsample'
        )
```

- Number of Estimators: 150 trees in the forest.

- Maximum Depth: 4 levels to prevent overfitting.

- Minimum Samples for Splitting: At least 8 samples required per node.

- Minimum Samples per Leaf: Each leaf must have at least 4 samples for stability.

- Class Balancing: balanced_subsample adjusts class weights within each tree.


#### **Common Spatial Patterns**

CSP is a signal processing technique widely used in EEG analysis to improve classification performance. It is commonly applied in Brain-Computer Interfaces (BCI) to distinguish between different mental states, such as imagining movement of the left hand versus the right hand.

***How CSP Works***

- Spatial Filtering: CSP applies a transformation to EEG signals using a combination of electrode channels.

- Decomposition into Components: The method finds spatial patterns that maximize variance differences between two classes.

- Feature Extraction: CSP selects the most relevant components that help differentiate brain activity states.

In [12]:
def training_cv_pipeline(X_train, y_train, transformer_name="CSP", n_folds=5):
    """
    Perform cross-validation on the training set.
    """
    print(f"\n\033[1;35m🔀 FIND BEST PIPELINE with CROSS-VALIDATION ({n_folds} folds) \033[0m")
    
    # Create KFold cross-validator
    kf = KFold(n_splits=n_folds, shuffle=True)
    
    # Create transformers based on the specified type
    if transformer_name == "CSP":
        from mne.decoding import CSP as MNE_CSP
        transformer1 = MNE_CSP()
        transformer2 = MNE_CSP()
        transformer3 = MNE_CSP()
    elif transformer_name == "MY_CSP":
        transformer1 = CSP(n_components=6)  # LDA
        transformer2 = CSP(n_components=8)  # LogReg
        transformer3 = CSP(n_components=10) # RandomForest
    else:
        raise ValueError(f"Unknown transformer: {transformer_name}")
    
    # Define classifiers with regularization
    lda = LDA(solver='lsqr', shrinkage=0.7)
    log_reg = LogisticRegression(
        penalty='l1', solver='saga', C=0.2, max_iter=300, tol=1e-4, multi_class='auto', class_weight='balanced'
    )
    rfc = RandomForestClassifier(
        n_estimators=150, max_depth=4, min_samples_split=8, min_samples_leaf=4, class_weight='balanced_subsample'
    )
    
    # Create pipelines
    pipeline1 = make_pipeline(transformer1, lda)
    pipeline2 = make_pipeline(transformer2, log_reg)
    pipeline3 = make_pipeline(transformer3, rfc)
    
    pipelines = [
        ('LDA', pipeline1),
        ('LOGR', pipeline2),
        ('RFC', pipeline3)
    ]
    
    best_pipeline = {'name': None, 'pipeline': None, 'cv_score': -1}
    
    # Perform cross-validation for each pipeline
    for name, pipeline in pipelines:
        cv_scores = []
        
        print(f"\nEvaluating {name} with {n_folds}-fold cross-validation...")
        
        # Manual cross-validation to get more detailed information
        fold_idx = 1
        for train_idx, val_idx in kf.split(X_train):
            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
            
            # Train on this fold
            pipeline.fit(X_fold_train, y_fold_train)
            
            # Evaluate on validation fold
            fold_score = pipeline.score(X_fold_val, y_fold_val)
            cv_scores.append(fold_score)
            
            print(f"  Fold {fold_idx}: {fold_score:.3f}")
            fold_idx += 1
        
        mean_cv_score = np.mean(cv_scores)
        std_cv_score = np.std(cv_scores)
        
        print(f"  {name} CV Score: {mean_cv_score:.3f} ± {std_cv_score:.3f}")
        
        # Check if this is the best pipeline so far
        if mean_cv_score > best_pipeline['cv_score']:
            best_pipeline = {
                'name': name,
                'pipeline': pipeline,
                'cv_score': mean_cv_score,
                'cv_scores': cv_scores
            }
    
    print(f"\n  💫 BEST PIPELINE from CV: {best_pipeline['name']} 💫")
    print(f"  Mean CV Score: {best_pipeline['cv_score']:.3f}")
    print(f"  Description: {best_pipeline['pipeline']}")
    
    return best_pipeline['pipeline'], best_pipeline['cv_scores'], best_pipeline['cv_score'], best_pipeline['name']


### 11. Predict result

In [13]:
def predict(X_test, y_test, subjectId, experiment_name, log=True):  # Changed to True by default
    PREDICT_MODEL = f"../models/model_subject_{subjectId}_{experiment_name}.joblib"
    try:
        trained_model = load(PREDICT_MODEL)
    except FileNotFoundError as e:
        raise Exception(f"File not found: {PREDICT_MODEL}")

    # Evaluate on TEST SET (previously unseen data)
    test_score = trained_model.score(X_test, y_test)
    print(f"\n\033[1;35m🔮 TEST SET EVALUATION for subject {subjectId} \033[0m")
    print(f"Test accuracy: {test_score:.3f}")
    
    # Detailed prediction breakdown if requested
    if log:
        scores = []
        print("\nDetailed predictions:")
        print("epoch_nb =  [prediction]    [truth]    equal?")
        print("---------------------------------------------")
        
        correct_predictions = 0
        total_predictions = X_test.shape[0]
        
        for n in range(total_predictions):
            pred = trained_model.predict(X_test[n:n + 1, :, :])[0]
            truth = y_test[n:n + 1][0]
            # Improve equality visualization
            is_equal = pred == truth
            if is_equal:
                correct_predictions += 1
                
            equal_symbol = "\033[1;32m✓\033[0m" if is_equal else "\033[1;31m✗\033[0m"
            
            # Always show prediction information
            print(f"epoch_{n:2} =      [{pred}]           [{truth}]        {equal_symbol}")
            
            scores.append(1 - np.abs(pred - truth))
        
        # Add a summary at the end
        accuracy = np.mean(scores).round(3)
        print(f"\nIndividual prediction accuracy: {accuracy:.3f} ({correct_predictions}/{total_predictions} = {int(accuracy * 100)}%)")
        
        # Overfitting warning
        if accuracy > 0.95:
            print("\n❗ WARNING: Accuracy is very high (>95%), possible overfitting.")
            print("   Consider techniques such as:")
            print("   - Using more training data")
            print("   - Increasing regularization")
            print("   - Reducing model complexity")
    
    return test_score


### 12. Get config params

In [14]:
def get_config():
    """Returns the default configuration with randomly selected subjects and experiment"""
    import random
    
    # Select 6 random subjects between 1 and 109, without repetitions
    random_subjects = random.sample(range(1, 110), 6)
    
    # Select a random experiment
    experiment_options = list(EXPERIMENTS.keys())
    random_experiment = random.choice(experiment_options)
    
    print(f"Randomly selected subjects: {random_subjects}")
    print(f"Randomly selected experiment: {random_experiment}")
    print(f"Experiment events: {EXPERIMENTS[random_experiment][0]} vs {EXPERIMENTS[random_experiment][1]}")
    
    config = {
        'SUBJECTS': random_subjects,  # Random subjects
        'TRANSFORMER': 'CSP',
        'EXPERIMENT': random_experiment  # Random experiment
    }

    return config


### 13. Processing subject

In [15]:
def process_subject(subjectID, args, isSingleSubject=False):
    start_time_inner = time.time()
    
    print(f"\n\033[1;33m⭐ Processing Subject {subjectID} ⭐\033[0m")
    
    try:
        # Try to load existing data first
        print(f"\n\033[1;35m📊 LOADING DATA for subject {subjectID}...\033[0m ")
        [X, y, epochs] = pre_process_data(subjectID, EXPERIMENTS[args['EXPERIMENT']])
        print(f"Data for subject {subjectID} loaded successfully!")
    except Exception as e:
        print(f"❌ Error processing subject {subjectID}: {str(e)}")
        # Try to download data automatically if it doesn't exist
        try:
            print(f"Trying to download data for subject {subjectID}...")
            for run in EXPERIMENTS_IDS['action'] + EXPERIMENTS_IDS['imagery']:
                downloaded_files = eegbci.load_data(subjectID, run, path=DATA_SAMPLE_PATH, force_update=True)
                print(f"  - Downloaded files for run {run}: {len(downloaded_files)} files")
            
            # Retry processing after download
            print(f"\n\033[1;35mRetrying to load data for subject {subjectID}...\033[0m ")
            [X, y, epochs] = pre_process_data(subjectID, EXPERIMENTS[args['EXPERIMENT']])
            print(f"Data for subject {subjectID} loaded successfully on second attempt!")
        except Exception as e2:
            print(f"❌ Fatal error with subject {subjectID}: {str(e2)}")
            return {
                'subject_id': subjectID,
                'error': str(e2),
                'pipelines': [],
                'cross_val_score': 0,
                'accuracy': 0,
                'time_cost': time.time() - start_time_inner
            }

    print(f"\n\033[1;35m💥 SPLIT DATASET for subject {subjectID} \033[0m")
    X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)
    
    stats = {
        'subject_id': subjectID,
        'pipelines': [],
        'cv_scores': {},
        'test_accuracy': 0
    }
    
    # if args['MODE'] == "train" or args['MODE'] == "all":
    best_pipeline, cv_scores, mean_cv_score, best_pipeline_name = training_cv_pipeline(
        X_train, y_train, transformer_name=args['TRANSFORMER'], n_folds=5
    )
    
    # Guardar estadísticas
    stats['cv_scores'] = [float(score) for score in cv_scores]
    stats['mean_cv_score'] = float(mean_cv_score)
    stats['best_pipeline'] = best_pipeline_name
    
    # Entrenar el modelo final con todos los datos de entrenamiento
    print(f"\n\033[1;35m💪 TRAINING FINAL MODEL for subject {subjectID} \033[0m")
    best_pipeline.fit(X_train, y_train)
    train_score = best_pipeline.score(X_train, y_train)
    
    print(f"Training score: {train_score:.3f}")
    stats['train_score'] = float(train_score)
    
    # Guardar el pipeline
    fileName = f"../models/model_subject_{subjectID}_{args['EXPERIMENT']}.joblib"
    dump(best_pipeline, fileName)
    print(f"Model saved to {fileName}")

    PREDICT_MODEL = f"../models/model_subject_{subjectID}_{args['EXPERIMENT']}.joblib"
    try:
        best_pipeline = load(PREDICT_MODEL)
    except FileNotFoundError:
        print(f"❌ Model file not found: {PREDICT_MODEL}")
        return stats
        
    print(f"\n\033[1;35m🔮 TEST SET EVALUATION for subject {subjectID} \033[0m")
    test_score = best_pipeline.score(X_test, y_test)
    print(f"Test accuracy: {test_score:.3f}")
    stats['test_accuracy'] = float(test_score)
    
    # Mostrar predicciones detalladas
    print("\nDetailed predictions:")
    print("epoch_nb =  [prediction]    [truth]    equal?")
    print("---------------------------------------------")
    
    correct_predictions = 0
    total_predictions = X_test.shape[0]
    
    for n in range(total_predictions):
        pred = best_pipeline.predict(X_test[n:n + 1, :, :])[0]
        truth = y_test[n:n + 1][0]
        is_equal = pred == truth
        if is_equal:
            correct_predictions += 1
            
        equal_symbol = "\033[1;32m✓\033[0m" if is_equal else "\033[1;31m✗\033[0m"
        print(f"epoch_{n:2} =      [{pred}]           [{truth}]        {equal_symbol}")
    
    # Agregar un resumen al final
    accuracy = correct_predictions / total_predictions
    print(f"\nIndividual prediction accuracy: {accuracy:.3f} ({correct_predictions}/{total_predictions} = {int(accuracy * 100)}%)")

    end_time_inner = time.time()
    time_cost_inner = end_time_inner - start_time_inner
    stats['time_cost'] = time_cost_inner
    print(f"Time cost: {round(stats['time_cost'], 2)} seconds")
    return stats


### 14. Stats

In [16]:
def calculate_all_means(cv_scores_list, test_scores, final_stats):
    print("\n\033[1;32mMean Scores for all subjects \033[0m")
    
    # Mejorado para mostrar un informe más detallado
    if cv_scores_list:
        # Tomar el promedio de los puntajes de CV para cada sujeto
        mean_cv_scores = [np.mean(scores) for scores in cv_scores_list]
        overall_mean_cv = np.mean(mean_cv_scores).round(3)
        overall_std_cv = np.std(mean_cv_scores).round(3)
        print(f"Mean cross-validation score: {overall_mean_cv:.3f} (std: {overall_std_cv:.3f})")
        final_stats['mean_cv_score'] = float(overall_mean_cv)
        final_stats['std_cv_score'] = float(overall_std_cv)
        
    if test_scores:
        mean_test = np.mean(test_scores).round(3)
        std_test = np.std(test_scores).round(3)
        print(f"Mean test accuracy:          {mean_test:.3f} (std: {std_test:.3f})")
        final_stats['mean_test_score'] = float(mean_test)
        final_stats['std_test_score'] = float(std_test)
        
    print("\n\033[1;32mIndividual Scores \033[0m")

    for i, (cv_scores, test_score) in enumerate(zip(cv_scores_list, test_scores)):
        subj_id = final_stats['subjects'][i]['subject_id']
        mean_cv = np.mean(cv_scores).round(3)
        print(f"Subject {subj_id:3}: Mean CV score = {mean_cv:.3f}, Test accuracy = {round(test_score, 3):.3f}")
    
    print("\n\033[1;32mCross-Validation Details \033[0m")
    print("Subject | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 | Mean CV")
    print("--------------------------------------------------------------")
    
    for i, cv_scores in enumerate(cv_scores_list):
        subj_id = final_stats['subjects'][i]['subject_id']
        fold_scores = cv_scores[:5] if len(cv_scores) >= 5 else cv_scores + [0] * (5 - len(cv_scores))
        mean_cv = np.mean(cv_scores).round(3)
        score_str = " | ".join([f"{score:.4f}" for score in fold_scores])
        print(f"{subj_id:7} | {score_str} | {mean_cv:.4f}")


def dump_result_to_json(final_stats, args):
    """
    Save the results to a JSON file with detailed information about each subject and
    aggregate statistics across all subjects.
    """
    # Create directory if it doesn't exist
    os.makedirs("../results", exist_ok=True)
    
    # Generate a clearer filename with timestamp
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    results_filename = \
        f"../results/results-{args['EXPERIMENT']}-{timestamp}.json"

    with open(results_filename, 'w', encoding='utf-8') as f:
        json.dump(final_stats, f, ensure_ascii=False, indent=4)

    # Count which pipelines were selected
    pipeline_counts = {}
    for subject in final_stats['subjects']:
        if 'best_pipeline' in subject:
            pipeline_name = subject['best_pipeline']
            pipeline_counts[pipeline_name] = pipeline_counts.get(pipeline_name, 0) + 1
    
    # Find the most chosen pipeline
    best_pipeline = None
    if pipeline_counts:
        best_pipeline = max(pipeline_counts, key=pipeline_counts.get)
        print(f"\nMost frequently selected pipeline: {best_pipeline} ({pipeline_counts[best_pipeline]} subjects)")

    # Generate and display a more complete summary
    print("\n===================== \033[1;32m FINAL SUMMARY \033[0m =====================")
    print(f"Experiment:  {args['EXPERIMENT']}")
    print(f"Transformer: {args['TRANSFORMER']}")
    print(f"Compared events: {EXPERIMENTS[args['EXPERIMENT']][0]} vs {EXPERIMENTS[args['EXPERIMENT']][1]}")
    print(f"Analyzed subjects: {args['SUBJECTS']}")
    print(f"Successfully processed subjects: {final_stats.get('successful_subjects', 0)}/{len(args['SUBJECTS'])}")
    
    if 'mean_cv_score' in final_stats:
        print(f"Average CV score: {final_stats['mean_cv_score']:.3f} (std: {final_stats.get('std_cv_score', 0):.3f})")
    if 'mean_test_score' in final_stats:
        print(f"Average test accuracy: {final_stats['mean_test_score']:.3f} (std: {final_stats.get('std_test_score', 0):.3f})")
    
    print(f"Total time: {round(final_stats['time_cost'], 2)} seconds")
    print(f"Complete results have been saved to:\n   [{results_filename}]")
    
    # Summary table by subject
    print("\n----------- Summary by Subject --------------")
    print("Subject | Pipeline | CV Score | Test Accuracy")
    print("----------------------------------------------")
    
    # Calculate averages for final row
    total_subjects = 0
    sum_cv_score = 0
    sum_test_score = 0
    pipeline_distribution = {}
    
    for subject in final_stats['subjects']:
        subject_id = subject['subject_id']
        
        # Skip subjects with errors
        if 'error' in subject:
            print(f"{subject_id:7} | ERROR    | ERROR    | ERROR")
            continue
            
        # Get pipeline name
        pipeline_name = subject.get('best_pipeline', 'N/A')
        if pipeline_name != 'N/A':
            pipeline_distribution[pipeline_name] = pipeline_distribution.get(pipeline_name, 0) + 1
            
        # Get mean CV score if available
        cv_score = 0
        if 'cv_scores' in subject:
            cv_score = np.mean(subject['cv_scores'])
        elif 'mean_cv_score' in subject:
            cv_score = subject['mean_cv_score']
            
        test_accuracy = subject.get('test_accuracy', 0)
        
        # Only count valid subjects for the average
        total_subjects += 1
        sum_cv_score += cv_score
        sum_test_score += test_accuracy
            
        print(f"{subject_id:7} | {pipeline_name:8} | {cv_score:.3f}    | {test_accuracy:.3f}")
    
    print("-------------------------------------------")
    
    # Add row with averages
    avg_cv = sum_cv_score / total_subjects if total_subjects > 0 else 0
    avg_test = sum_test_score / total_subjects if total_subjects > 0 else 0
    
    # Show most common pipeline if any
    most_common_pipeline = 'N/A'
    if pipeline_distribution:
        most_common_pipeline = max(pipeline_distribution, key=pipeline_distribution.get)
    
    print(f"AVERAGE |  ------  | \033[1;32m{avg_cv:.3f}\033[0m    | \033[1;32m{avg_test:.3f}\033[0m")
    
    # Show pipeline distribution
    if pipeline_distribution:
        print("\nPipeline Distribution:")
        for pipeline, count in pipeline_distribution.items():
            percentage = (count / total_subjects) * 100
            print(f"  {pipeline:4}: {count} subjects ({percentage:.1f}%)")
    
    print("==========================================================")


### 15. Utils

In [17]:
def ensure_directories():
    """Ensures that necessary directories exist"""
    
    # Required directories
    dirs = [
        "../data/raw/files",
        "../models",
        "../results",
    ]
    
    for directory in dirs:
        os.makedirs(directory, exist_ok=True)


def reduce_mne_verbosity():
    """
    Reduce verbosity of MNE loggers by setting them to WARNING level.
    This will suppress the detailed INFO messages during CSP computation.
    """
    # Set MNE logger to WARNING level (less verbose)
    mne_logger = logging.getLogger('mne')
    mne_logger.setLevel(logging.WARNING)
    
    # Set scipy logger to WARNING level (also can be verbose)
    scipy_logger = logging.getLogger('scipy')
    scipy_logger.setLevel(logging.WARNING)
    
    # Set matplotlib logger to WARNING level
    mpl_logger = logging.getLogger('matplotlib')
    mpl_logger.setLevel(logging.WARNING)
    
    print("MNE and related loggers set to WARNING level (less verbose)")


### 16. Evaluation

In [18]:
def evaluate():
    print("\n==========================================")
    print("\033[1;36m     EEG DATA ANALYSIS CLASSIFICATION \033[0m")
    print("==========================================\n")
    
    # Reduce verbosity of MNE and related libraries
    reduce_mne_verbosity()

    start_time = time.time()
    
    # Ensure necessary directories exist
    ensure_directories()
    
    # Use default configuration
    args = get_config()
    print("Configuration:", args)

    print(f"Experiment under study: ({EXPERIMENTS[args['EXPERIMENT']][0]}) <--VS--> ({EXPERIMENTS[args['EXPERIMENT']][1]})")
    print(f"Transformer: {args['TRANSFORMER']}")
    # print(f"Mode: {args['MODE']}")
    print(f"Subjects to process: {args['SUBJECTS']}\n\n")
    
    CALC_MEAN_FOR_ALL = True if len(args['SUBJECTS']) > 1 else False

    cv_scores_list = []
    test_scores = []
    final_stats = {
        'subjects_hash': "all" if len(args['SUBJECTS']) == 109 else ''.join(map(str, args['SUBJECTS'])),
        'config': args,
        'events': EXPERIMENTS[args['EXPERIMENT']],
        'subjects': [],
        'time_unit': "seconds",
        'total_subjects': len(args['SUBJECTS']),
        'successful_subjects': 0
    }

    for subjectID in args['SUBJECTS']:
        result = process_subject(subjectID, args, isSingleSubject=not CALC_MEAN_FOR_ALL)
        
        # Only add valid results
        if 'error' not in result:
            cv_scores_list.append(result['cv_scores'])

            test_scores.append(result['test_accuracy'])
                
            final_stats['successful_subjects'] += 1

        final_stats['subjects'].append(result)
        
        # Show progress
        print(f"\n🏁 Progress: {final_stats['subjects'].index(result)+1}/{len(args['SUBJECTS'])} subjects processed\n")

    if CALC_MEAN_FOR_ALL and cv_scores_list:  # Only calculate mean if there are valid results
        calculate_all_means(cv_scores_list, test_scores, final_stats)

    final_stats['time_cost'] = time.time() - start_time

    dump_result_to_json(final_stats, args)


evaluate()



     EEG DATA ANALYSIS CLASSIFICATION 

MNE and related loggers set to WARNING level (less verbose)
Randomly selected subjects: [89, 80, 86, 44, 41, 42]
Randomly selected experiment: imagery_vs_action__feets
Experiment events: do/feet vs imagine/feet
Configuration: {'SUBJECTS': [89, 80, 86, 44, 41, 42], 'TRANSFORMER': 'CSP', 'EXPERIMENT': 'imagery_vs_action__feets'}
Experiment under study: (do/feet) <--VS--> (imagine/feet)
Transformer: CSP
Subjects to process: [89, 80, 86, 44, 41, 42]



⭐ Processing Subject 89 ⭐

📊 LOADING DATA for subject 89... 
Class distribution: {0: 22, 2: 24}
Data for subject 89 loaded successfully!

💥 SPLIT DATASET for subject 89 
Original dataset shape: (46, 64, 801)
Original class distribution: {0: 22, 2: 24}
Dataset is already balanced.
Dataset shape after balancing: (46, 64, 801)
Train set shape:  (36, 64, 801)
Test set shape:   (10, 64, 801)
Train set class distribution: 	{0: 17, 2: 19}
Test set class distribution: 	{0: 5, 2: 5}

🔀 FIND BEST PIPELINE with 

/sgoinfre/students/dgerwig-/miniforge/envs/vortex_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


  Fold 3: 0.857
  Fold 4: 0.857
  Fold 5: 1.000
  LOGR CV Score: 0.868 ± 0.080

Evaluating RFC with 5-fold cross-validation...
  Fold 1: 1.000
  Fold 2: 0.750
  Fold 3: 0.714
  Fold 4: 0.857
  Fold 5: 0.857
  RFC CV Score: 0.836 ± 0.100

  💫 BEST PIPELINE from CV: LOGR 💫
  Mean CV Score: 0.868
  Description: Pipeline(steps=[('csp', CSP()),
                ('logisticregression',
                 LogisticRegression(C=0.2, class_weight='balanced',
                                    max_iter=300, penalty='l1',
                                    solver='saga'))])

💪 TRAINING FINAL MODEL for subject 42 
Training score: 0.865
Model saved to ../models/model_subject_42_imagery_vs_action__feets.joblib

🔮 TEST SET EVALUATION for subject 42 
Test accuracy: 1.000

Detailed predictions:
epoch_nb =  [prediction]    [truth]    equal?
---------------------------------------------
epoch_ 0 =      [0]           [0]        ✓
epoch_ 1 =      [2]           [2]        ✓
epoch_ 2 =      [2]           [2]   